# 03 · Monitoreo y evidencias para el README

Ejecuta cada celda y captura la salida. Si una tabla de sistema no está disponible, la celda muestra el error (va al README) y se usa la alternativa.

In [ ]:
dbutils.widgets.text("catalogo", "electrocasa_dev", "Catálogo del ambiente")
CAT = dbutils.widgets.get("catalogo")

def intentar(sql, titulo):
    print("=====", titulo)
    try:
        display(spark.sql(sql))
    except Exception as e:
        print("ERROR:", str(e).splitlines()[0][:500])

## 1. Event log del pipeline (actualizaciones y métricas de expectativas)

In [ ]:
intentar(f"""
SELECT timestamp, event_type, message
FROM event_log(TABLE({CAT}.silver.ventas))
WHERE event_type IN ('create_update', 'update_progress')
ORDER BY timestamp DESC
LIMIT 30
""", "Actualizaciones del pipeline")

intentar(f"""
WITH exp AS (
  SELECT timestamp,
         explode(from_json(details:flow_progress:data_quality:expectations,
                 'array<struct<name:string, dataset:string, passed_records:bigint, failed_records:bigint>>')) AS e
  FROM event_log(TABLE({CAT}.silver.ventas))
  WHERE event_type = 'flow_progress'
)
SELECT e.dataset, e.name AS expectativa,
       sum(e.passed_records) AS filas_ok, sum(e.failed_records) AS filas_fallidas
FROM exp
GROUP BY ALL
ORDER BY dataset, expectativa
""", "Métricas de expectativas por tabla")

## 2. Historial de corridas del job (`system.lakeflow`)

In [ ]:
intentar("""
WITH j AS (
  SELECT workspace_id, job_id, name
  FROM system.lakeflow.jobs
  QUALIFY row_number() OVER (PARTITION BY workspace_id, job_id ORDER BY change_time DESC) = 1
)
SELECT j.name, t.run_id, t.period_start_time, t.period_end_time, t.result_state, t.trigger_type
FROM system.lakeflow.job_run_timeline t
JOIN j USING (workspace_id, job_id)
WHERE j.name LIKE '%electrocasa_job%'
  AND t.result_state IS NOT NULL
ORDER BY t.period_start_time DESC
LIMIT 50
""", "Corridas del job")

## 3. Costos (`system.billing.usage`, reto opcional)

In [ ]:
intentar("""
SELECT usage_date,
       coalesce(usage_metadata.job_name, usage_metadata.dlt_pipeline_id, usage_metadata.job_id, sku_name) AS recurso,
       billing_origin_product,
       round(sum(usage_quantity), 4) AS dbus
FROM system.billing.usage
WHERE usage_date >= DATE'2026-11-18'
GROUP BY ALL
ORDER BY usage_date DESC, dbus DESC
""", "Consumo por recurso")

## 4. Conector de Tracking: dos corridas (la segunda no agrega versión)

In [ ]:
intentar(f"DESCRIBE HISTORY {CAT}.bronze.tracking_envios", "Historial de bronze.tracking_envios")
intentar(f"SELECT count(*) AS eventos, min(fecha_modificacion), max(fecha_modificacion) FROM {CAT}.bronze.tracking_envios",
         "Eventos ingeridos")

## 5. COPY INTO idempotente

In [ ]:
intentar(f"DESCRIBE HISTORY {CAT}.bronze.catalogo", "Historial de bronze.catalogo")

## 6. Masking: un miembro de `electrocasa_ingenieria` ve los valores reales también en la tabla derivada de Gold

Ejecútalo con tu usuario (miembro de ingeniería) y, si puedes, con un usuario fuera del grupo para mostrar la diferencia.

In [ ]:
intentar("SELECT current_user() AS usuario, is_account_group_member('electrocasa_ingenieria') AS en_ingenieria", "Usuario")
intentar(f"SELECT id_empleado, nombre, dni, salario, sucursal_id FROM {CAT}.silver.empleados_historia LIMIT 5", "Silver (columnas con MASK)")
intentar(f"SELECT id_empleado, nombre, dni, salario, sucursal_id FROM {CAT}.gold.empleados_vista LIMIT 5", "Gold derivada (columnas con MASK)")
intentar(f"DESCRIBE TABLE EXTENDED {CAT}.gold.empleados_vista", "Definición (columnas con MASK)")

## 7. SCD2: ejemplo de un empleado transferido y ubicación de eventos tardíos

In [ ]:
intentar(f"""
SELECT id_empleado, version, tipo_evento, sucursal_id, estado, fecha_evento, fecha_registro,
       vigente_desde, vigente_hasta, es_version_actual
FROM {CAT}.silver.empleados_historia
WHERE id_empleado IN (SELECT id_empleado FROM {CAT}.silver.empleados_eventos
                      WHERE tipo_evento = 'TRANSFERENCIA' LIMIT 3)
ORDER BY id_empleado, version
""", "Versiones de empleados transferidos")

## 8. VARIANT en reseñas: estructura real de `metadata`

In [ ]:
intentar(f"SELECT schema_of_variant_agg(metadata) AS esquema FROM {CAT}.silver.resenas", "Esquema de metadata")
intentar(f"SELECT * FROM {CAT}.gold.resenas_negativas_por_origen ORDER BY total_resenas DESC", "Negativas por origen")

## 9. Exploración para el contrato (variantes encontradas en Bronze)

In [ ]:
intentar(f"SELECT metodo_pago, count(*) n FROM {CAT}.bronze.ventas GROUP BY 1 ORDER BY 2 DESC", "metodo_pago (Bronze)")
intentar(f"SELECT metodo_pago_original, metodo_pago, count(*) n FROM {CAT}.silver.ventas GROUP BY ALL ORDER BY 2", "metodo_pago (mapeo)")
intentar(f"SELECT categoria, count(*) n FROM {CAT}.bronze.catalogo GROUP BY 1 ORDER BY 2 DESC", "categoria (Bronze)")
intentar(f"SELECT tipo_evento, count(*) n FROM {CAT}.bronze.empleados_eventos GROUP BY 1 ORDER BY 2 DESC", "tipo_evento (Bronze)")
intentar(f"SELECT estado_entrega, count(*) n FROM {CAT}.bronze.tracking_envios GROUP BY 1 ORDER BY 2 DESC", "estado_entrega (Bronze)")
intentar(f"SELECT courier, count(*) n FROM {CAT}.bronze.tracking_envios GROUP BY 1 ORDER BY 2 DESC", "courier (Bronze)")
intentar(f"SELECT canal, count(*) n FROM {CAT}.bronze.ventas GROUP BY 1 ORDER BY 2 DESC", "canal (Bronze)")
intentar(f"""SELECT fuente, regla, politica, id_registro, registro
FROM {CAT}.silver.cuarentena WHERE regla IN ('categoria_valida', 'metodo_pago_conocido', 'tipo_evento_valido', 'estado_valido')
LIMIT 50""", "Valores que quedaron fuera del mapeo (ajustar utils/normalizacion.py)")